## Import and Connections

In [1]:
import redshift_connector

# Create a connection to the Redshift database with the above credentials
conn = redshift_connector.connect(
    host='c23-workgroup.129033205317.eu-west-2.redshift-serverless.amazonaws.com',
    database='dw_air_travel',
    user='yusuf_ahmed',
    password='Dyusuf_71',
    port=5439
)
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')

# Extract the result from the cursor object
output = cursor.fetch_dataframe()

print(output)

# Close the cursor
cursor.close()

   count
0    112


In [2]:
import numpy as np
from scipy.stats import zscore
import pandas as pd
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [4]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
        SELECT
            laser_incident_id,
            aircraft,
            altitude,
            laser_colour,
            injury,
            airport_id,
            at
        FROM s_yusuf_ahmed.laser_incident
        WHERE at >= '2020-01-01'
          AND at < '2022-01-01'
    """)

        laser_df = cursor.fetch_dataframe()
        print("Number of rows:", len(michigan_laser))

    except Exception as e:
        conn.rollback()
        print(e)

Number of rows: 11052


## How did the frequency/severity of laser incidents change between the two years?

In [7]:
laser_df['at'] = pd.to_datetime(laser_df['at'])
laser_df['year'] = laser_df['at'].dt.year
laser_df['month'] = laser_df['at'].dt.to_period('M').astype(str)

incidents_by_year = (
    laser_df.groupby('year')
    .size()
    .reset_index(name='incident_count')
)

incidents_by_year.head()

,year,incident_count
0,2020,6251
1,2021,4801


### Incidents by year

In [ ]:
alt.Chart(incidents_by_year).mark_bar().encode(
    x=alt.X('year', title='Year'),
    y=alt.Y('incident_count', title='Number of Incidents'),
    tooltip=['year', 'incident_count']
).properties(
    width=200,
    height=350,
    title='Laser Incidents by Year'
)

alt.Chart(...)

### Incident by month

In [14]:
incidents_by_month = (
    laser_df.groupby('month')
    .size()
    .reset_index(name='incident_count')
)

alt.Chart(incidents_by_month).mark_line(point=True).encode(
    x=alt.X('month', title='Month'),
    y=alt.Y('incident_count', title='Number of Incidents'),
    tooltip=['month', 'incident_count']
).properties(
    width=800,
    height=400,
    title='Laser Incidents Over Time'
)

alt.Chart(...)

### Injury severity by year

In [16]:
laser_df['injury_status'] = laser_df['injury'].map({'t': 1, 'f': 0})

severity_by_month = (
    laser_df.groupby('month')
    .agg(
        incidents=('laser_incident_id', 'count'),
        injury_count=('injury_status', 'sum'),
        injury_rate=('injury_status', 'mean')
    )
    .reset_index()
)

severity_by_month['injury_rate_pct'] = severity_by_month['injury_rate'] * 100

In [17]:
alt.Chart(severity_by_month).mark_bar().encode(
    x=alt.X('month', title='Month'),
    y=alt.Y('injury_rate_pct', title='Injury Rate (%)'),
    tooltip=[
        'month',
        'incidents',
        'injury_count',
        alt.Tooltip('injury_rate_pct', format='.2f', title='Injury Rate (%)')
    ]
).properties(
    width=500,
    height=350,
    title='Severity by month (Injury Rate)'
)

alt.Chart(...)

## What types of aircraft are more often affected by laser incidents?

In [18]:
laser_df['aircraft'].value_counts().head(30)

aircraft
C172       1075
B737        835
E75L        522
B738        494
HELO        446
A320        356
P28A        295
B763        272
CRJ2        225
B739        225
A321        220
E145        219
CRJ9        216
unknown     214
PC12        207
CRJ7        185
B752        179
C208        178
DA40        156
E170        147
A319        144
A306        140
AS50        101
C182         93
MD11         91
BE9L         90
E75S         86
SR22         85
B767         80
EC35         79
Name: count, dtype: int64

In [19]:
def aircraft_category(code):
    code = str(code).upper()

    # Helicopters
    if code in ['HELO'] or code.startswith('AS') or code.startswith('EC'):
        return 'Helicopter'

    # General aviation (small aircraft)
    elif code.startswith(('C1', 'C2', 'P', 'SR', 'DA', 'PC')):
        return 'General Aviation'

    # Boeing
    elif code.startswith('B7'):
        return 'Commercial (Boeing) Plane'

    # Airbus
    elif code.startswith('A3'):
        return 'Commercial (Airbus) Plane'

    # Regional jets
    elif code.startswith(('CRJ', 'E')):
        return 'Regional Jet'

    # Unknown
    elif code == 'UNKNOWN':
        return 'Unknown'

    else:
        return 'Other'

laser_df['aircraft_type'] = laser_df['aircraft'].apply(aircraft_category)

In [20]:
aircraft_summary =(
    laser_df.groupby('aircraft_type')
    .agg(
        incidents=('laser_incident_id', 'count'),
        injury_count=('injury_status', 'sum'),
        injury_rate=('injury_status', 'mean')
    )
    .reset_index()
)

aircraft_summary['injury_rate_pct'] = aircraft_summary['injury_rate'] * 100

In [26]:
alt.Chart(aircraft_summary).mark_bar().encode(
    y=alt.Y('incidents:Q', title='Number of Incidents'),
    x=alt.X('aircraft_type:N', sort='-y', title = 'Aircraft Type'),
    tooltip=[
        'aircraft_type',
        'incidents',
        alt.Tooltip('injury_rate_pct:Q', format='.2f', title='Injury Rate (%)')
    ]
).properties(
    width=600,
    height=300,
    title='Laser Incidents by Aircraft Type'
)

alt.Chart(...)